# Análises de Zika — esquema estrela unificado

Execute após **um** dos pipelines. Ambos publicam o mesmo esquema MySQL.
As consultas e visualizações ficam neste arquivo, separadas da engenharia dos pipelines.
2026 é parcial; contagens descrevem notificações, não incidência nem risco populacional.
Gráficos são exportados para `saidas/analises/`.

In [ ]:
from pathlib import Path
import sys

# Execute na raiz do clone ou em notebooks/ (inclusive no Colab).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pipelines/contrato.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Clone o repositório e abra o notebook dentro dele.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
# Crie previamente um banco dedicado ao projeto, usando utf8mb4.
# Configure a conexão antes de executar. A senha vem do ambiente, sem ser salva no notebook.
os.environ.setdefault('MYSQL_HOST', '127.0.0.1')
os.environ.setdefault('MYSQL_PORT', '3306')
os.environ.setdefault('MYSQL_USER', 'root')
os.environ.setdefault('MYSQL_DATABASE', 'zika_dw')
# os.environ['MYSQL_PASSWORD'] = getpass.getpass('Senha MySQL: ')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pipelines.database import connect
output = ROOT / 'saidas/analises'
output.mkdir(parents=True, exist_ok=True)

def query(sql):
    with connect() as connection, connection.cursor() as cursor:
        cursor.execute(sql)
        return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)


## 1. Notificações mensais e sazonalidade

Esta análise investiga como o volume de notificações de Zika se distribui ao longo dos meses entre 2024 e 2026, avaliando a concentração sazonal no primeiro semestre e a variação entre os anos em períodos comparáveis.

#### Dados Importantes:

- Concentração sazonal (Jan–Mai): 70,8% (58.307 de 82.404) das notificações do triênio ocorreram entre janeiro e maio, com picos em março de 2024 (10.366 registros) e abril de 2025 (3.829 registros).
- Variação anual (Jan–Set): Como a base de 2026 é parcial (vai até 05/10/2026), comparando apenas os nove meses completos de janeiro a setembro, houve queda contínua de 41,7% entre 2024 e 2025 (39.373 para 22.942) e de 41,7% entre 2025 e 2026 (22.942 para 13.384)

#### Conclusões:

As notificações concentram-se fortemente nos cinco primeiros meses de cada ano e apresentam redução consistente nos períodos comparáveis do triênio, lembrando que os dados refletem registros notificados e não taxas de incidência populacional.

No Brasil, o período de janeiro a maio reúne temperaturas elevadas e chuvas frequentes no Sudeste, Centro-Oeste, Norte e litoral nordestino. A água parada acelera o ciclo ovo-adulto do mosquito (que cai para cerca de 7 a 10 dias no calor) e aumenta a densidade vetorial, atingindo o ápice de transmissão entre fevereiro e abril.

In [ ]:
monthly = query("""SELECT t.ano, t.mes, CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes
FROM fato_notificacao_zika f JOIN dim_tempo t ON t.sk_tempo=f.fk_tempo_notificacao
GROUP BY t.ano,t.mes ORDER BY t.ano,t.mes""")
ax = monthly.pivot(index='mes', columns='ano', values='notificacoes').plot(marker='o')
ax.set(xlabel='Mês', ylabel='Notificações', title='Série mensal (2026 parcial)')
plt.tight_layout()
plt.savefig(output / '01_sazonalidade.png')
plt.show()
display(monthly)

## 2. Gestantes por UF de residência

Esta análise verifica a distribuição estadual das notificações em gestantes e compara o volume total de casos suspeitos com a parcela que efetivamente teve diagnóstico confirmado por unidade federativa.

#### Dados Importantes:

- Total e concentração de notificações: Foram registradas 7.184 gestantes (8,72% do total), das quais 207 (2,88%) foram confirmadas. Cinco estados concentram 67,4% das notificações: SP (2.487), ES (736), PE (631), MS (590) e MG (397).
- Destaque nas confirmações: Enquanto SP confirmou apenas 0,12% (3 de 2.487) das gestantes notificadas, Mato Grosso (MT) reuniu 52,7% de todas as gestantes confirmadas do país (109 de 207; taxa de 33,23%).

#### Conclusões:

Os estados com maior volume de notificações suspeitas em gestantes não são os mesmos que concentram os casos confirmados, evidenciando diferenças regionais na triagem, investigação e encerramento laboratorial.

Em SP, a vigilância é guiada pelo pré-natal: Mais da metade de todas as fichas de SP (55,50%) são gestantes com sintomas leves ou manchas na pele notificadas preventivamente em 225 municípios para descartar infecção congênita. Como o vírus quase não circulava no estado, 97,3% dessas gestantes testaram negativo.

Em MT, a notificação reflete demanda clínica real: Apenas 5,07% das fichas são gestantes; os outros 94,93% (6.147 casos) são pacientes da população geral que procuraram atendimento com quadro agudo durante surtos locais.

In [ ]:
pregnant = query("""SELECT l.sigla_uf, CAST(SUM(f.fl_gestante) AS SIGNED) AS gestantes,
CAST(SUM(f.fl_gestante * f.fl_confirmado) AS SIGNED) AS confirmadas
FROM fato_notificacao_zika f JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
GROUP BY l.sigla_uf ORDER BY gestantes DESC""")
ax = pregnant.set_index('sigla_uf')[['gestantes','confirmadas']].plot.bar(figsize=(12,4))
ax.set(xlabel='UF de residência (NI = não informada)', ylabel='Notificações', title='Gestantes notificadas e confirmadas')
plt.tight_layout()
plt.savefig(output / '02_gestantes.png')
plt.show()
display(pregnant)

## 3. Tempo até notificação e encerramento

Esta análise mede a agilidade da vigilância epidemiológica nas regiões brasileiras, avaliando o tempo médio entre o início dos sintomas e a notificação, bem como o prazo entre a notificação e o encerramento do caso.

#### Dados Importantes:

- Atraso até a notificação: A média nacional foi de 5,69 dias (mediana de 3,0 dias; 100% de cobertura válida), variando de 4,49 dias no Centro-Oeste a 10,13 dias no Sul.
- Tempo até o encerramento: A média nacional foi de 28,99 dias (mediana de 19,0 dias; 97,67% de cobertura válida), com o menor tempo médio no Norte (23,37 dias) e o maior no Sudeste (32,55 dias).

#### Conclusões:

O registro inicial da suspeita ocorre em poucos dias na maior parte do país, mas a conclusão da investigação leva cerca de quatro semanas em média, sendo puxada para cima por uma parcela de casos com acompanhamento prolongado.

Encerramento laboratorial é duas vezes mais rápido que o clínico-epidemiológico: Quando o resultado do exame sai, a unidade fecha a ficha rapidamente (média de 22,14 dias; mediana de 15 dias). Já os casos sem exame conclusivo ficam aguardando evolução clínica e fechamento epidemiológico por quase o dobro do tempo (43,83 dias).

Diferença regional na composição: No Norte, 79,1% dos casos são encerrados por critério laboratorial e apenas 14,8% por critério clínico-epidemiológico — puxando a média regional para baixo (23,37 dias). No Sudeste, a proporção de encerramento clínico-epidemiológico sobe para 28,3% (quase o dobro do Norte) e as gestantes representam 24,8% das notificações (contra 4,0% no Norte). Como gestantes exigem acompanhamento pré-natal mais longo antes de fechar o caso (média de 32,13 dias contra 28,76 dias nos demais pacientes), a média do Sudeste sobe para 32,55 dias.

In [ ]:
delay = query("""SELECT l.regiao_brasil, AVG(f.qt_dias_atraso_notif) AS atraso_notificacao,
AVG(f.qt_dias_encerramento) AS tempo_encerramento
FROM fato_notificacao_zika f JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
GROUP BY l.regiao_brasil ORDER BY l.regiao_brasil""")
ax = delay.set_index('regiao_brasil').astype(float).plot.barh()
ax.set(xlabel='Dias (média dos intervalos válidos de 0 a 365)', ylabel='Região', title='Tempos da vigilância')
plt.tight_layout()
plt.savefig(output / '03_tempos.png')
plt.show()
display(delay)

## 4. Perfil Sociodemográfico por Faixa Etária e Sexo

Esta análise examina como as notificações e confirmações de Zika se distribuem por sexo e faixa etária, identificando os grupos populacionais mais presentes nos registros da vigilância.

#### Dados Importantes:

- Predomínio feminino na idade fértil: As mulheres respondem por 60,1% (49.559 de 82.404) de todas as notificações, com maior diferença entre 15 e 44 anos (64,7% do total nessa faixa), impulsionada pela busca ativa em gestantes.
- Inversão na infância e maior confirmação em idosos: Entre 0 e 14 anos, o sexo masculino é maioria (52,4%; 8.393 contra 7.635), enquanto a maior taxa de confirmação ocorre entre mulheres com 60 anos ou mais (7,52%; 375 de 4.988).

#### Conclusões:

O perfil das notificações é fortemente influenciado pelo monitoramento materno-infantil em mulheres adultas jovens, ao passo que a positividade proporcional dos casos é mais elevada na população idosa.

In [ ]:
profile = query("""SELECT p.faixa_etaria, p.descricao_sexo,
CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes,
CAST(SUM(f.fl_confirmado) AS SIGNED) AS confirmados
FROM fato_notificacao_zika f JOIN dim_paciente p ON p.sk_paciente=f.fk_paciente
GROUP BY p.faixa_etaria, p.descricao_sexo ORDER BY p.faixa_etaria, p.descricao_sexo""")

bands = ['0 a 14 anos', '15 a 29 anos', '30 a 44 anos', '45 a 59 anos', '60 anos ou mais']
ax = (profile[profile['descricao_sexo'].isin(['Feminino', 'Masculino'])]
      .pivot(index='faixa_etaria', columns='descricao_sexo', values='notificacoes')
      .reindex(bands).plot.bar())
ax.set(xlabel='Faixa etária', ylabel='Notificações', title='Notificações por faixa etária e sexo')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(output / '04_perfil_demografico.png')
plt.show()
profile.to_csv(tables_out / '04_perfil_faixa_etaria_sexo.csv', index=False)
display(profile)

## 5. Classificação Final e Critério de Confirmação por Ano

Esta análise avalia a qualidade e o desfecho das investigações epidemiológicas entre 2024 e 2026, verificando quantos casos confirmados tiveram respaldo laboratorial e qual a proporção de registros ainda sem conclusão.

#### Dados Importantes:

- Dos 4.619 casos confirmados no triênio, 79,7% (3.683) tiveram confirmação laboratorial e 19,3% (891) vínculo clínico-epidemiológico, com o peso laboratorial subindo de 78,2% em 2024 para 82,8% em 2026.
- Casos em aberto: A parcela de notificações classificadas como Inconclusivo / Em Investigação passou de 7,0% em 2024 e 6,9% em 2025 para 11,0% (1.465 de 13.385) em 2026.

#### Conclusões:

A confirmação diagnóstica da Zika no período apoia-se majoritariamente em exames laboratoriais, mas o ano mais recente ainda concentra maior proporção de fichas aguardando encerramento.

In [ ]:
outcome = query("""SELECT t.ano, d.classificacao_final, d.criterio_confirmacao,
CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes
FROM fato_notificacao_zika f
JOIN dim_tempo t ON t.sk_tempo=f.fk_tempo_notificacao
JOIN dim_investigacao_desfecho d ON d.sk_desfecho=f.fk_desfecho
GROUP BY t.ano, d.classificacao_final, d.criterio_confirmacao
ORDER BY t.ano, d.classificacao_final, notificacoes DESC""")

ax = (outcome[outcome['classificacao_final'] == 'Confirmado']
      .pivot(index='ano', columns='criterio_confirmacao', values='notificacoes')
      .plot.bar(stacked=True))
ax.set(xlabel='Ano (2026 parcial)', ylabel='Casos confirmados', title='Confirmados por critério e ano')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(output / '05_criterio_desfecho.png')
plt.show()
outcome.to_csv(tables_out / '05_classificacao_criterio_ano.csv', index=False)
display(outcome)

## 6. Deslocamento Assistencial entre Residência e Unidade Notificadora

Esta análise cruza o local de moradia do paciente com o local da unidade de saúde que registrou o caso, medindo com que frequência o atendimento ocorre fora do município ou do estado de residência.

#### Dados Importantes:

- Atendimento fora do município: Entre os 82.372 registros com município válido, 8,25% (6.793) foram notificados fora da cidade de residência, com destaque para o Sul (17,11%) e o Sudeste (13,24%), contra apenas 3,39% no Centro-Oeste.
- Atendimento fora do estado: Apenas 0,66% (546 registros) ocorreram fora da UF de origem, mas a região Norte concentra 56,0% de todos esses deslocamentos interestaduais (306 de 546 casos; taxa de 2,00% na região).

#### Conclusões:

Embora a grande maioria das notificações ocorra no próprio município do paciente, há fluxos relevantes de deslocamento intermunicipal no Sul/Sudeste e de busca por atendimento em estados vizinhos na região Norte.

In [ ]:
mobility = query("""SELECT l.regiao_brasil,
ROUND(AVG(l.cod_municipio_ibge6 <> u.cod_municipio_notificacao) * 100, 2) AS fora_municipio,
ROUND(AVG(l.sigla_uf <> u.sigla_uf_notificacao) * 100, 2) AS fora_uf
FROM fato_notificacao_zika f
JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
JOIN dim_unidade_notificadora u ON u.sk_unidade=f.fk_unidade_notificadora
WHERE l.cod_municipio_ibge6 IS NOT NULL AND u.cod_municipio_notificacao IS NOT NULL
GROUP BY l.regiao_brasil ORDER BY fora_municipio DESC""")

ax = mobility.set_index('regiao_brasil').astype(float).plot.barh()
ax.set(xlabel='Percentual de notificações (%)', ylabel='Região', title='Atendimento fora do município e UF de residência')
plt.tight_layout()
plt.savefig(output / '06_mobilidade_assistencial.png')
plt.show()
mobility.to_csv(tables_out / '06_mobilidade_assistencial_regiao.csv', index=False)
display(mobility)